# fasthtml

> In-kernel live inspector panel served via FastHTML

In [ ]:
#| default_exp fasthtml

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
import json
from urllib.parse import quote
from fasthtml.common import to_xml
from starlette.testclient import TestClient
import paar.bridge as B, paar.fasthtml as F
from paar.fasthtml import app, _node, _broadcast, _acc
from paar.core import VarInfo

class _IP:
    def __init__(self, ns): self.user_ns=ns; self.user_ns_hidden=set()
    class events:
        @staticmethod
        def register(n, fn): pass

def test_node_scalar_no_toggle():
    html = to_xml(_node(VarInfo(name='n', type='int', value='5', accessor=('n',), path='n')))
    assert 'n' in html and 'int' in html and '5' in html
    assert '/expand' not in html and '<details' not in html
def test_node_container_details():
    html = to_xml(_node(VarInfo(name='d', type='dict', value='{...}',
                                is_container=True, accessor=('d',), path='d')))
    assert '<details' in html and '<summary' in html
    assert 'paar-children' in html and '/expand?accessor=' in html
def test_node_container_lazy_once():
    html = to_xml(_node(VarInfo(name='d', type='dict', is_container=True, accessor=('d',), path='d')))
    assert 'once' in html   # hx-trigger="click once" -> children fetched only on first open
def test_node_more_sentinel():
    html = to_xml(_node(VarInfo(name='…', type='', value='150 more', accessor=('L',), path='L', more_offset=100)))
    assert '/expand?accessor=' in html and 'offset=100' in html and '150 more' in html
    assert 'hx-target="this"' in html and '<details' not in html   # clicking swaps itself for the next page
def test_rows_route_lists_vars():
    B.get_ipython = lambda: _IP({'alpha': 123})
    r = TestClient(app).get('/rows?profile=standard')
    assert r.status_code==200 and 'alpha' in r.text and '123' in r.text and 'id="rows"' in r.text
def test_home_wires_ws_and_loader():
    r = TestClient(app).get('/')
    assert r.status_code==200
    assert 'ws-connect="/live"' in r.text and 'hx-ext="ws"' in r.text
    assert 'id="rows"' in r.text and '/rows' in r.text
def test_home_has_profile_select():
    r = TestClient(app).get('/')
    assert '<select' in r.text and 'name="profile"' in r.text
    assert 'minimal' in r.text and 'standard' in r.text   # profile options rendered
def test_rows_profile_filters_and_groups():
    import math
    B.get_ipython = lambda: _IP({'myvar': 5, 'mymod': math})
    mn = TestClient(app).get('/rows?profile=minimal').text
    assert 'myvar' in mn and 'Modules' not in mn          # minimal hides the module group
    st = TestClient(app).get('/rows?profile=standard').text
    assert 'myvar' in st and 'Modules' in st and 'mymod' in st   # standard groups modules
def test_expand_route_returns_children():
    B.get_ipython = lambda: _IP({'d': {'x': 1}})
    r = TestClient(app).get('/expand?accessor=' + quote(json.dumps(['d'])))
    assert r.status_code==200 and 'x' in r.text and '1' in r.text
def test_expand_route_paging():
    B.get_ipython = lambda: _IP({'L': list(range(250))})
    r = TestClient(app).get('/expand?accessor=' + quote(json.dumps(['L'])) + '&offset=100')
    assert r.status_code==200 and '100' in r.text and 'offset=200' in r.text   # trailing load-more points at page 3
def test_broadcast_drops_dead_clients():
    F._clients.clear()
    def _bad_send(_): raise RuntimeError('dead')
    F._clients.append(('not-a-loop', _bad_send))
    _broadcast('x')
    assert F._clients==[]
def test_node_grid_only_toggles_grid():
    html = to_xml(_node(VarInfo(name='arr', type='ndarray', is_grid=True, accessor=('arr',), path='arr')))
    assert '<details' in html and '/grid?accessor=' in html and 'paar-gridbox' in html
    assert '/expand' not in html   # not a container -> the row toggles the grid, no tree
def test_node_grid_and_container_shows_both():
    html = to_xml(_node(VarInfo(name='arr', type='ndarray', is_grid=True, is_container=True,
                                accessor=('arr',), path='arr')))
    assert '/expand?accessor=' in html and '/grid?accessor=' in html   # tree + grid coexist
    assert 'paar-gridbox' in html and 'paar-children' in html and 'paar-gridtoggle' in html
    assert html.count('<details') >= 2 and html.count('click once') >= 2   # both are lazy toggles
def test_grid_route_renders_table():
    import numpy as np
    B.get_ipython = lambda: _IP({'arr': np.arange(6).reshape(2,3)})
    r = TestClient(app).get('/grid?accessor=' + quote(json.dumps(['arr'])) + '&roff=0&coff=0')
    assert r.status_code==200 and '<table' in r.text and '5' in r.text  # last cell value present
for t in [test_node_scalar_no_toggle,test_node_container_details,test_node_container_lazy_once,
          test_node_more_sentinel,test_rows_route_lists_vars,test_home_wires_ws_and_loader,
          test_home_has_profile_select,test_rows_profile_filters_and_groups,
          test_expand_route_returns_children,test_expand_route_paging,
          test_broadcast_drops_dead_clients,test_node_grid_only_toggles_grid,
          test_node_grid_and_container_shows_both,test_grid_route_renders_table]: t()

In [ ]:
# JSON API routes (the frontend-agnostic feed consumed by the terminal frontend, paar.tui)
import os, tempfile
os.environ['PAAR_REG_DIR'] = tempfile.mkdtemp()   # keep registry writes out of ~/.paar during tests

def test_api_rows_json():
    import math
    B.get_ipython = lambda: _IP({'alpha': 123, 'mymod': math})
    r = TestClient(app).get('/api/rows?profile=standard')
    assert r.status_code==200
    j = r.json()
    assert j['profile']=='standard' and 'standard' in j['profiles']
    names = [n['name'] for g in j['groups'] for n in g['nodes']]
    assert 'alpha' in names
    assert 'Modules' in [g['label'] for g in j['groups']]   # standard groups modules
def test_api_rows_profile_independent():
    B.get_ipython = lambda: _IP({'x': 1})
    j = TestClient(app).get('/api/rows?profile=minimal').json()
    assert j['profile']=='minimal'   # explicit profile honored without mutating server state
def test_api_expand_json():
    B.get_ipython = lambda: _IP({'d': {'x': 1}})
    j = TestClient(app).get('/api/expand?accessor=' + quote(json.dumps(['d']))).json()
    assert any(n['name']=="'x'" for n in j) and any(n['value']=='1' for n in j)
def test_api_expand_paging_sentinel():
    B.get_ipython = lambda: _IP({'L': list(range(250))})
    j = TestClient(app).get('/api/expand?accessor=' + quote(json.dumps(['L']))).json()
    assert j[-1]['more_offset']==100   # trailing load-more sentinel carries the next offset
def test_api_grid_json():
    import numpy as np
    B.get_ipython = lambda: _IP({'arr': np.arange(6).reshape(2,3)})
    j = TestClient(app).get('/api/grid?accessor=' + quote(json.dumps(['arr']))).json()
    assert j['nrows']==2 and j['ncols']==3 and j['cells'][1][2]=='5'
def test_api_envs_json():
    j = TestClient(app).get('/api/envs').json()
    assert 'env' in j and isinstance(j['envs'], list)   # discovery feed shape
for t in [test_api_rows_json, test_api_rows_profile_independent, test_api_expand_json,
          test_api_expand_paging_sentinel, test_api_grid_json, test_api_envs_json]: t()

In [ ]:
# serve(): headless launcher wires an explicit namespace + registers the env (server thread stubbed out)
def test_serve_wires_ns_and_registry():
    class _Fake:
        def __init__(self, app, port, daemon): self.port=port
    orig, F._server = F.JupyUvi, None
    F.JupyUvi = _Fake
    try:
        url = F.serve(port=8765, ns={'sv': 1}, name='unit', poll=999)
        assert url.endswith(':8765') and F._env=='unit'
        rows = TestClient(app).get('/api/rows').json()
        assert 'sv' in [n['name'] for g in rows['groups'] for n in g['nodes']]   # inspects the passed ns
        assert 'unit' in [e['name'] for e in F.R.active()]                        # registered for discovery
        assert 'unit' in TestClient(app).get('/api/envs').text
    finally:
        F.JupyUvi = orig; F.R.unregister(F._port)
        F._server = F._port = F._env = None; F.set_namespace(None)
test_serve_wires_ns_and_registry()

In [ ]:
#| export
import asyncio, threading, json, sys, time, atexit, argparse
from urllib.parse import quote
from fasthtml.common import *
from fasthtml.jupyter import JupyUvi, HTMX
from paar.bridge import Bridge, on_change, set_namespace
from paar.core import VarInfo
from paar.snapshot import PROFILES
import paar.registry as R

In [ ]:
#| export
_hljs = (   # inline python syntax-highlighting; theme follows the OS/browser color scheme
    Link(rel='stylesheet', media='(prefers-color-scheme: dark)',
         href='https://cdn.jsdelivr.net/gh/highlightjs/cdn-release/build/styles/atom-one-dark.min.css'),
    Link(rel='stylesheet', media='(prefers-color-scheme: light)',
         href='https://cdn.jsdelivr.net/gh/highlightjs/cdn-release/build/styles/atom-one-light.min.css'),
    Script(src='https://cdn.jsdelivr.net/gh/highlightjs/cdn-release/build/highlight.min.js'),
    Script("htmx.onLoad(el=>el.querySelectorAll('code.pv:not([data-highlighted])')"
           ".forEach(c=>hljs.highlightElement(c)))"))
_css = Style(
    '.paar-node{font-size:.8rem;line-height:1.45;'
    'font-family:ui-monospace,SFMono-Regular,Menlo,monospace} '
    '.paar-children{margin-left:.55rem;padding-left:.4rem;'
    'border-left:1px solid var(--pico-muted-border-color)} '
    'details.paar-node{margin:0} '
    'summary{cursor:pointer;list-style:none;display:block;margin:0;padding:0} '
    'summary::-webkit-details-marker{display:none} '
    'summary::before{content:"\\25B8";display:inline-block;width:1em;opacity:.45} '
    'details[open]>summary::before{content:"\\25BE"} '
    '.paar-leaf{padding-left:1em} '
    '.paar-name{color:var(--pico-primary)} '
    '.paar-eq{opacity:.4} '
    '.paar-type{opacity:.5;font-size:.72rem} '
    '.paar-node code{white-space:pre;background:none;padding:0} '
    'code.hljs{display:inline;background:none;padding:0} '
    '.paar-group>summary{font-weight:600;opacity:.65;text-transform:uppercase;'
    'font-size:.7rem;letter-spacing:.03em} '
    '.paar-profile{font-size:.75rem;padding:.1rem .3rem;margin:0 0 .4rem;width:auto} '
    '.paar-gridbox:not(:empty){margin:.25rem 0 .25rem .85rem} '
    '.paar-grid{max-height:360px;overflow:auto} '
    '.paar-grid table{font-size:.8rem;margin:0} '
    '.paar-grid th{position:sticky;top:0;background:var(--pico-background-color)} '
    '.paar-gridnav{margin:.25rem 0} '
    '.paar-page{margin-right:.5rem;cursor:pointer} '
    '.paar-grid-label{opacity:.6;cursor:pointer}')
bridge = Bridge()
app,rt = fast_app(exts='ws', hdrs=(_css, *_hljs))   # ws ext + hljs
_clients = []   # list[(loop, send)]
_clients_lock = threading.Lock()
_server = None
_port = None    # this server's bound port
_env = None     # this server's env label, shown in the UI + registry (set by serve()/inspector())
_profile = 'standard'   # active inspector profile (mutated by the /rows dropdown)

In [ ]:
#| export
def _acc(accessor):
    "URL-safe encoding of a positional accessor."
    return quote(json.dumps(list(accessor)), safe='')

def _head(v:VarInfo):
    "PyCharm-style row: name = {type: shape} value, with a syntax-highlighted value."
    typ = f'{v.type}: {v.shape}' if v.shape else v.type
    return Span(Span(v.name, cls='paar-name'), Span(' = ', cls='paar-eq'),
                Span('{', typ, '}', cls='paar-type', title=v.qualifier), ' ',
                Code(v.value, cls='pv language-python'),
                Small(f' {v.dtype}', cls='paar-type') if v.dtype else '',
                cls=('error' if v.is_error else None))

def _grid_toggle(v):
    "A collapsible '▦ grid' sub-node that lazily loads the paged table on first open."
    return Details(
        Summary(Span('▦ grid', cls='paar-grid-label'),
                hx_get=f'/grid?accessor={_acc(v.accessor)}&roff=0&coff=0',
                hx_target='next .paar-gridbox', hx_swap='innerHTML', hx_trigger='click once'),
        Div(cls='paar-gridbox'), cls='paar-node paar-gridtoggle')

def _more(v:VarInfo):
    "The load-more sentinel: clicking it swaps itself for the next page of children."
    return Div(Span(f'… {v.value}', cls='paar-grid-label'),
               hx_get=f'/expand?accessor={_acc(v.accessor)}&offset={v.more_offset}',
               hx_target='this', hx_swap='outerHTML', hx_trigger='click',
               cls='paar-node paar-leaf paar-more-row')

def _node(v:VarInfo):
    "Render a tree node: the load-more sentinel fetches the next page; containers expand; gridables also get a collapsible grid; scalars are plain."
    if v.more_offset is not None: return _more(v)
    head = _head(v)
    if v.is_container:
        body = [_grid_toggle(v)] if v.is_grid else []   # grid toggle sits above the tree children
        return Details(
            Summary(head, hx_get=f'/expand?accessor={_acc(v.accessor)}',
                    hx_target='next .paar-children', hx_swap='innerHTML', hx_trigger='click once'),
            *body,
            Div(cls='paar-children'), cls='paar-node')
    if v.is_grid:   # gridable but not otherwise expandable: the row itself toggles the grid
        return Details(
            Summary(head, ' ', Span('▦ grid', cls='paar-grid-label'),
                    hx_get=f'/grid?accessor={_acc(v.accessor)}&roff=0&coff=0',
                    hx_target='next .paar-gridbox', hx_swap='innerHTML', hx_trigger='click once'),
            Div(cls='paar-gridbox'), cls='paar-node')
    return Div(head, cls='paar-node paar-leaf')

def _grid_nav(page, acc):
    "Prev/next paging controls + window info for a grid page."
    roff, coff = page['roff'], page['coff']
    npr, npc = len(page['index']), len(page['headers'])
    nrows, ncols = page['nrows'], page['ncols']
    pr, pc = page['rows'], page['cols']
    def lnk(label, ro, co):
        return A(label, hx_get=f'/grid?accessor={_acc(acc)}&roff={ro}&coff={co}',
                 hx_target='closest .paar-gridbox', hx_swap='innerHTML', cls='paar-page')
    ctrls = []
    if roff > 0: ctrls.append(lnk('◀ rows', max(0, roff-pr), coff))
    if roff+npr < nrows: ctrls.append(lnk('rows ▶', roff+npr, coff))
    if coff > 0: ctrls.append(lnk('◀ cols', roff, max(0, coff-pc)))
    if coff+npc < ncols: ctrls.append(lnk('cols ▶', roff, coff+npc))
    info = Small(f'rows {roff}-{roff+npr} of {nrows}, cols {coff}-{coff+npc} of {ncols}')
    return Div(info, ' ', *ctrls, cls='paar-gridnav')

def _grid(page, acc):
    "Render a grid page as a scrollable table with paging controls."
    if page is None: return Div('not gridable')
    hdr = Tr(Th(''), *[Th(h) for h in page['headers']])
    body = [Tr(Th(ix), *[Td(c) for c in row]) for ix, row in zip(page['index'], page['cells'])]
    return Div(Div(Table(Thead(hdr), Tbody(*body)), cls='paar-grid'),
               _grid_nav(page, acc))

def _group(label, vs):
    "A collapsed category group (Special Variables / Functions / Modules) holding its member nodes."
    return Details(Summary(Span(label)),
                   Div(*[_node(v) for v in vs], cls='paar-children'),
                   cls='paar-node paar-group')

def _profile_select():
    "The profile dropdown; on change GETs /rows for the chosen profile."
    return Select(*[Option(p, value=p, selected=(p==_profile)) for p in PROFILES],
                  name='profile', hx_get='/rows', hx_target='#rows', hx_swap='outerHTML',
                  hx_trigger='change', cls='paar-profile')

def _rows_div():
    "The active profile's grouped node tree, wrapped in the #rows target."
    nodes = []
    for label, vs in bridge.view(_profile):
        if label is None: nodes += [_node(v) for v in vs]
        else: nodes.append(_group(label, vs))
    return Div(*nodes, id='rows')

def _loader(oob=False):
    "A #rows div that immediately GETs /rows (used initially and as the WS nudge)."
    kw = dict(hx_get='/rows', hx_trigger='load', hx_swap='outerHTML')
    if oob: kw['hx_swap_oob']='true'
    return Div(id='rows', **kw)

In [ ]:
#| export
def _env_select():
    "Dropdown of live paar envs; picking one navigates the browser to that env's inspector. None if <2 envs."
    envs = R.active()
    if len(envs) < 2: return None
    here = f'http://127.0.0.1:{_port}' if _port else ''
    return Select(*[Option(f"{e['name']}  :{e['port']}", value=e['base'], selected=(e['base']==here))
                    for e in envs],
                  onchange='location.href=this.value', cls='paar-profile', title='environment')

In [ ]:
#| export
from starlette.responses import JSONResponse
from dataclasses import asdict

def _vd(v):
    "VarInfo -> JSON-safe dict (the accessor tuple serializes as a list)."
    return asdict(v)

@rt('/api/rows')
def api_rows(profile:str=None):
    "Grouped node tree for a profile as JSON — the frontend-agnostic feed the terminal UI (paar.tui) consumes."
    p = profile if profile in PROFILES else _profile
    groups = [{'label':lbl, 'nodes':[_vd(v) for v in vs]} for lbl,vs in bridge.view(p)]
    return JSONResponse({'profile':p, 'profiles':list(PROFILES), 'groups':groups})

@rt('/api/expand')
def api_expand(accessor:str, offset:int=0):
    "One level of children at accessor as a JSON list of VarInfo dicts (trailing load-more sentinel carries more_offset)."
    return JSONResponse([_vd(v) for v in bridge.expand(tuple(json.loads(accessor)), offset)])

@rt('/api/grid')
def api_grid(accessor:str, roff:int=0, coff:int=0):
    "A grid page at accessor as JSON (headers/index/cells/nrows/ncols/…), or null if not gridable."
    return JSONResponse(bridge.grid(tuple(json.loads(accessor)), roff, coff))

@rt('/api/envs')
def api_envs():
    "This server's env label + all live paar environments — the discovery feed for the frontend env picker."
    return JSONResponse({'env':_env, 'envs':R.active()})

In [ ]:
#| export
@rt('/')
def home():
    "Inspector page: env picker (when >1 env is live) + profile select + the live-updating node tree."
    title = f'paar · {_env}' if _env else 'paar'
    kids = [_env_select(), _profile_select(),
            Div(_loader(), hx_ext='ws', ws_connect='/live', id='paar')]
    return Titled(title, *[k for k in kids if k is not None])

@rt('/rows')
def rows(profile:str=None):
    global _profile
    if profile in PROFILES: _profile = profile
    return _rows_div()

@rt('/expand')
def expand_route(accessor:str, offset:int=0):
    # flat fragment: innerHTML on first open, or outerHTML replacing the load-more sentinel on a page click
    return tuple(_node(v) for v in bridge.expand(tuple(json.loads(accessor)), offset))

@rt('/grid')
def grid_route(accessor:str, roff:int=0, coff:int=0):
    acc = tuple(json.loads(accessor))
    return _grid(bridge.grid(acc, roff, coff), acc)

def _drop(send):
    "Remove a client by its send identity (thread-safe)."
    with _clients_lock:
        _clients[:] = [(l,s) for (l,s) in _clients if s is not send]

async def _conn(send):
    with _clients_lock: _clients.append((asyncio.get_running_loop(), send))
async def _disconn(send): _drop(send)

@app.ws('/live', conn=_conn, disconn=_disconn)
async def live(send): pass

In [ ]:
#| export
def _broadcast(fragment):
    "Push fragment to every WS client from any thread; drop clients that fail."
    with _clients_lock: targets = list(_clients)
    for loop, send in targets:
        try: fut = asyncio.run_coroutine_threadsafe(send(fragment), loop)
        except Exception: _drop(send); continue
        fut.add_done_callback(lambda f, s=send: (not f.cancelled()) and f.exception() is not None and _drop(s))

def _sig():
    "Cheap fingerprint of the active profile view — changes when a displayed name/type/value/shape/dtype changes."
    return tuple((v.name, v.type, v.value, v.shape, v.dtype)
                 for _, vs in bridge.view(_profile) for v in vs)

def _poll_loop(interval):
    "Background thread: broadcast an OOB refresh whenever the namespace fingerprint changes (headless path)."
    last = None
    while True:
        time.sleep(interval)
        try: cur = _sig()
        except Exception: continue
        if cur != last: last = cur; _broadcast(to_xml(_loader(oob=True)))

def inspector(port=8000, height=520, profile='standard', name=None):
    "Start the in-kernel live inspector panel (Jupyter/IPython) and return the inline iframe."
    global _server, _port, _env, _profile
    _profile = profile; _env = R.env_name(name)
    if _server is None:
        _port = port
        _server = JupyUvi(app, port=port, daemon=True)
        R.register(_env, port); atexit.register(R.unregister, port)
        on_change(lambda: _broadcast(to_xml(_loader(oob=True))))
    return HTMX(port=port, height=height, link=True)

def serve(port=8000, ns=None, name=None, poll=0.5, profile='standard'):
    "Start the paar inspector in a background thread for a plain (non-Jupyter) process; returns its base URL.\n\n    ns defaults to the caller's module globals; name defaults to the repo/package/cwd label. Refresh is driven\n    by a poll+diff thread, so any other environment can point paar-tui (or a browser) at the returned URL."
    global _server, _port, _env, _profile
    if ns is None: ns = sys._getframe(1).f_globals
    set_namespace(ns); _profile = profile; _env = R.env_name(name, ns)
    if _server is None:
        _port = R.free_port(port)
        _server = JupyUvi(app, port=_port, daemon=True)
        base = f'http://127.0.0.1:{_port}'
        R.register(_env, _port, base); atexit.register(R.unregister, _port)
        threading.Thread(target=_poll_loop, args=(poll,), daemon=True).start()
    return f'http://127.0.0.1:{_port}'

def serve_main(argv=None):
    "Console entry point: `paar-serve script.py [args...]` — run a script under a live paar inspector, then keep the server up."
    import os.path as osp
    p = argparse.ArgumentParser(prog='paar-serve', description='run a python script under a live paar inspector')
    p.add_argument('script', help='python script to execute and inspect live')
    p.add_argument('--port', type=int, default=8000, help='base port (auto-bumps if taken)')
    p.add_argument('--name', default=None, help='env label shown in the UI (default: script name)')
    p.add_argument('rest', nargs=argparse.REMAINDER, help='args passed through to the script')
    a = p.parse_args(argv)
    ns = {'__name__': '__main__', '__file__': a.script}
    base = serve(port=a.port, ns=ns, name=a.name or osp.splitext(osp.basename(a.script))[0])
    print(f'paar inspector live at {base}   (connect a terminal with: paar-tui)')
    sys.argv = [a.script, *a.rest]
    try:
        with open(a.script) as f: code = compile(f.read(), a.script, 'exec')
        exec(code, ns)
    finally:                      # script done: hold the process so its final namespace stays inspectable
        print('script finished — inspector still live, press Ctrl-C to stop')
        try:
            while True: time.sleep(1)
        except KeyboardInterrupt: pass

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()